# Contextual Subspace VQE + OpenFermion Pipeline

## Overview

This notebook implements a clean pipeline that:

1. **Generates a molecular Hamiltonian** using OpenFermion given a molecular geometry
2. **Converts data formats** between OpenFermion and cs_vqe (Kirby et al.) representations
3. **Splits the Hamiltonian** into **noncontextual** and **contextual** parts using CS-VQE
4. **Converts the contextual (reduced) Hamiltonian** back to OpenFermion `FermionOperator` and `QubitOperator` formats for downstream use

---

## Data Type Flow

```
Geometry (list of tuples)
       │
       ▼  [OpenFermion / PySCF]
FermionOperator  ←── fermion_ham
       │
       ▼  [jordan_wigner()]
QubitOperator    ←── qubit_ham
       │
       ▼  [qubit_op_to_csvqe_dict()]
dict {Pauli_str: coeff}  ←── cs_vqe 'ham'
       │
       ▼  [cs_vqe.greedy_dfs()]
dict {Pauli_str: coeff}  ←── ham_noncon  (noncontextual part)
dict {Pauli_str: coeff}  ←── ham_con     (contextual part = ham - ham_noncon)
       │
       ▼  [csvqe_dict_to_qubit_op() + reverse_jw()]
QubitOperator    ←── con_qubit_ham  (reduced qubit Hamiltonian)
FermionOperator  ←── con_fermion_ham (approximate inverse JW)
```

### Key format details

| Format | Description | Example |
|--------|-------------|---------|
| `FermionOperator` | OpenFermion fermionic second-quantized operator | `0.5 [0^ 1] + 0.5 [1^ 0]` |
| `QubitOperator` | OpenFermion qubit Pauli operator | `0.25 [X0 Y1] + ...` |
| cs_vqe `ham` dict | Pauli strings → real coefficients, string length = n_qubits | `{'IXYZ': 0.5, 'IIII': 1.0}` |

**Note:** The cs_vqe library (Kirby 2021) expects Pauli strings where:
- Each character is `'I'`, `'X'`, `'Y'`, or `'Z'`
- String length equals the total number of qubits
- Index 0 corresponds to the leftmost character
- Coefficients must be **real**

This is the reverse of OpenFermion's `QubitOperator` convention (index 0 = first qubit, acts from right).
In cs_vqe, `'XYZ'` on 3 qubits means X on qubit 0, Y on qubit 1, Z on qubit 2 — **same ordering** as OpenFermion internally, so the conversion is straightforward.


## 0. Installation & Imports

In [1]:
# Install dependencies if needed
# !pip install openfermion openfermionpyscf
# Clone CS-VQE library:
# !git clone https://github.com/wmkirby1/ContextualSubspaceVQE.git

import sys
import os
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Add path to ContextualSubspaceVQE repo — adjust this path!
CSVQE_PATH = './ContextualSubspaceVQE'  # <-- set this to wherever you cloned the repo
sys.path.insert(1, CSVQE_PATH)

import cs_vqe as c

import openfermion as of
from openfermion import (
    FermionOperator,
    QubitOperator,
    MolecularData,
    jordan_wigner,
    reverse_jordan_wigner,
    get_sparse_operator,
    count_qubits,
)

# If using openfermionpyscf for real molecule generation:
try:
    from openfermionpyscf import run_pyscf
    PYSCF_AVAILABLE = True
    print("openfermionpyscf available — will use PySCF for molecule generation.")
except ImportError:
    PYSCF_AVAILABLE = False
    print("openfermionpyscf not available — will use a built-in H2 Hamiltonian for demonstration.")

print(f"OpenFermion version: {of.__version__}")

openfermionpyscf available — will use PySCF for molecule generation.
OpenFermion version: 1.6.1


---
## 1. Format Conversion Utilities

These two helper functions bridge the OpenFermion and cs_vqe worlds. Define them once and reuse throughout.

In [2]:
def qubit_op_to_csvqe_dict(qubit_op: QubitOperator, n_qubits: int) -> dict:
    """
    Convert an OpenFermion QubitOperator to the cs_vqe Hamiltonian dict format.

    cs_vqe format: {Pauli_string: float_coefficient}
    where Pauli_string has fixed length n_qubits, characters in {I, X, Y, Z}.
    e.g. QubitOperator('X0 Z2', 0.5) on 4 qubits -> {'XIZI': 0.5}

    Parameters
    ----------
    qubit_op : QubitOperator
        The OpenFermion qubit operator (from jordan_wigner or similar).
    n_qubits : int
        Total number of qubits. Must be >= count_qubits(qubit_op).

    Returns
    -------
    dict
        Mapping from Pauli strings to real coefficients.

    Notes
    -----
    - Imaginary parts of coefficients are dropped (warn if non-negligible).
    - Duplicate Pauli strings are summed.
    - The identity term () in QubitOperator maps to 'III...I'.
    """
    ham_dict = {}
    for term, coeff in qubit_op.terms.items():
        # term is a tuple of (qubit_index, pauli_char) pairs, e.g. ((0,'X'), (2,'Z'))
        pauli_chars = ['I'] * n_qubits
        for qubit_idx, pauli_char in term:
            if qubit_idx >= n_qubits:
                raise ValueError(
                    f"Qubit index {qubit_idx} exceeds n_qubits={n_qubits}. "
                    f"Increase n_qubits or recheck your operator."
                )
            pauli_chars[qubit_idx] = pauli_char
        pauli_str = ''.join(pauli_chars)

        # Warn if imaginary part is non-negligible
        if abs(coeff.imag) > 1e-10:
            warnings.warn(
                f"Term '{pauli_str}' has imaginary coefficient {coeff}. "
                f"cs_vqe requires real coefficients — discarding imaginary part."
            )
        real_coeff = coeff.real

        ham_dict[pauli_str] = ham_dict.get(pauli_str, 0.0) + real_coeff

    # Remove zero-coefficient terms
    ham_dict = {k: v for k, v in ham_dict.items() if abs(v) > 1e-14}
    return ham_dict


def csvqe_dict_to_qubit_op(ham_dict: dict) -> QubitOperator:
    """
    Convert a cs_vqe Hamiltonian dict back to an OpenFermion QubitOperator.

    cs_vqe format: {Pauli_string: float_coefficient}
    e.g. {'XIZI': 0.5} -> QubitOperator('X0 Z2', 0.5)

    Parameters
    ----------
    ham_dict : dict
        Mapping from Pauli strings to real coefficients (cs_vqe format).

    Returns
    -------
    QubitOperator
        The corresponding OpenFermion QubitOperator.

    Notes
    -----
    - Identity-only strings like 'IIII' map to QubitOperator('') (scalar).
    - All Pauli strings must have the same length.
    """
    qubit_op = QubitOperator()
    for pauli_str, coeff in ham_dict.items():
        # Build OpenFermion term string from the Pauli characters
        term_parts = []
        for qubit_idx, pauli_char in enumerate(pauli_str):
            if pauli_char != 'I':
                term_parts.append(f'{pauli_char}{qubit_idx}')
        term_str = ' '.join(term_parts)  # '' for all-identity
        qubit_op += QubitOperator(term_str, coeff)
    return qubit_op


def compute_exact_ground_state_energy(qubit_op: QubitOperator) -> float:
    """
    Compute the exact ground state energy of a QubitOperator by
    sparse diagonalization. Only feasible for small systems (< ~20 qubits).

    Parameters
    ----------
    qubit_op : QubitOperator
    
    Returns
    -------
    float : ground state energy (minimum eigenvalue)
    """
    from scipy.sparse.linalg import eigsh
    n_q = count_qubits(qubit_op)
    sparse_mat = get_sparse_operator(qubit_op, n_qubits=n_q)
    eigenvalues, _ = eigsh(sparse_mat, k=1, which='SA')
    return float(eigenvalues[0].real)


print("Conversion utilities defined.")

Conversion utilities defined.


---
## 2. Step 1 — Generate `fermion_ham` and `qubit_ham` from Geometry

We support two paths:
- **Path A (recommended):** Use `openfermionpyscf` to run a real SCF/FCI calculation for any molecule
- **Path B (fallback):** Use an analytic H₂ Hamiltonian if PySCF is unavailable

**Output of this step:**
- `fermion_ham` — `FermionOperator`: second-quantized electronic Hamiltonian
- `qubit_ham` — `QubitOperator`: qubit representation via Jordan-Wigner transformation
- `n_qubits` — `int`: number of qubits = 2 × number of molecular orbitals

In [3]:
# ─── CONFIGURE YOUR MOLECULE HERE ─────────────────────────────────────────────

# Molecular geometry: list of (symbol, (x, y, z)) in Angstrom
geometry = [
    ('H', (0.0, 0.0, 0.0)),
    ('H', (0.0, 0.0, 0.74)),   # H2 at equilibrium bond length
]

basis = 'sto-3g'       # Basis set string (PySCF convention)
multiplicity = 1       # Spin multiplicity (2S+1): 1=singlet, 2=doublet, 3=triplet
charge = 0             # Net charge of the molecule

# For larger molecules, change geometry e.g.:
# geometry = [('Li', (0., 0., 0.)), ('H', (0., 0., 1.595))]  # LiH
# geometry = [('O', (0., 0., 0.)), ('H', (0.757, 0.586, 0.)), ('H', (-0.757, 0.586, 0.))]  # H2O

# ──────────────────────────────────────────────────────────────────────────────

if PYSCF_AVAILABLE:
    # ── Path A: Real molecule via PySCF ──────────────────────────────────────
    print("Running PySCF calculation...")

    molecule = MolecularData(
        geometry=geometry,
        basis=basis,
        multiplicity=multiplicity,
        charge=charge,
        description='cs_vqe_pipeline'
    )

    # Run RHF + FCI
    molecule = run_pyscf(
        molecule,
        run_scf=True,
        run_fci=True,
    )

    # Get the molecular Hamiltonian (InteractionOperator) and convert to FermionOperator
    molecular_ham = molecule.get_molecular_hamiltonian()
    fermion_ham = of.get_fermion_operator(molecular_ham)

    n_qubits = 2 * molecule.n_orbitals  # spin-orbitals = 2 × spatial orbitals
    print(f"Molecule: {molecule.name}")
    print(f"Nuclear repulsion energy: {molecule.nuclear_repulsion:.6f} Ha")
    print(f"HF energy:                {molecule.hf_energy:.6f} Ha")
    print(f"FCI ground state energy:  {molecule.fci_energy:.6f} Ha")
    print(f"Number of spin-orbitals:  {n_qubits}")

else:
    # ── Path B: Analytic H2/STO-3G Hamiltonian (no PySCF needed) ─────────────
    # Coefficients from standard JW-transformed H2 at R=0.74 Å, STO-3G basis.
    # This is the canonical 4-qubit benchmark Hamiltonian.
    print("Using analytic H2/STO-3G Hamiltonian (no PySCF).")

    # Reference values (Hartree):
    # HF energy:  -1.1175 Ha  |  FCI energy: -1.1373 Ha

    # Standard form of the H2 qubit Hamiltonian in Jordan-Wigner encoding
    qubit_ham_analytic = (
          QubitOperator('', -0.0988639693749)         # identity / nuclear repulsion + one-body
        + QubitOperator('Z0',  0.1714128264)          # Z0
        + QubitOperator('Z1', -0.2234315367)          # Z1
        + QubitOperator('Z2',  0.1714128264)          # Z2
        + QubitOperator('Z3', -0.2234315367)          # Z3
        + QubitOperator('Z0 Z1',  0.1686889820)       # ZZ
        + QubitOperator('Z0 Z2',  0.1200533213)       # ZZ
        + QubitOperator('Z0 Z3',  0.1659278503)       # ZZ
        + QubitOperator('Z1 Z2',  0.1659278503)       # ZZ
        + QubitOperator('Z1 Z3',  0.1743484408)       # ZZ
        + QubitOperator('Z2 Z3',  0.1686889820)       # ZZ
        + QubitOperator('X0 X1 Y2 Y3', -0.0453026155) # XXYY
        + QubitOperator('X0 Y1 Y2 X3',  0.0453026155) # XYYX
        + QubitOperator('Y0 X1 X2 Y3',  0.0453026155) # YXXY
        + QubitOperator('Y0 Y1 X2 X3', -0.0453026155) # YYXX
    )

    # Reconstruct a FermionOperator via reverse Jordan-Wigner
    fermion_ham = reverse_jordan_wigner(qubit_ham_analytic)
    n_qubits = 4

    print(f"H2 STO-3G: n_qubits = {n_qubits}")
    print(f"Number of terms in qubit Hamiltonian: {len(qubit_ham_analytic.terms)}")

# Apply Jordan-Wigner to get QubitOperator
qubit_ham = jordan_wigner(fermion_ham)
qubit_ham.compress()

print(f"\nFermionOperator: {len(fermion_ham.terms)} terms")
print(f"QubitOperator:   {len(qubit_ham.terms)} terms, {count_qubits(qubit_ham)} active qubits")

Running PySCF calculation...
Molecule: H2_sto-3g_singlet_cs_vqe_pipeline
Nuclear repulsion energy: 0.715104 Ha
HF energy:                -1.116759 Ha
FCI ground state energy:  -1.137284 Ha
Number of spin-orbitals:  4

FermionOperator: 37 terms
QubitOperator:   15 terms, 4 active qubits


---
## 3. Step 2 — Convert `qubit_ham` → cs_vqe dict format

**Why this conversion is needed:**  
cs_vqe uses a flat `dict` where keys are fixed-length Pauli strings and values are real floats.  
OpenFermion's `QubitOperator` uses a `dict` where keys are tuples of `(qubit_idx, pauli_char)` pairs.
The `qubit_op_to_csvqe_dict` function handles:
- Expanding sparse qubit indices into full-length strings (filling with `'I'`)
- Extracting real parts (JW-transformed molecular Hamiltonians are always Hermitian and real in the computational basis)

In [4]:
# Convert QubitOperator -> cs_vqe dict
ham = qubit_op_to_csvqe_dict(qubit_ham, n_qubits)

print(f"cs_vqe ham dict: {len(ham)} terms")
print(f"Pauli string length (= n_qubits): {len(next(iter(ham)))}")
print("\nFirst 10 terms:")
for i, (pauli_str, coeff) in enumerate(list(ham.items())[:10]):
    print(f"  '{pauli_str}': {coeff:.8f}")

# Sanity check: constant term (all-I string) should be present
identity_key = 'I' * n_qubits
if identity_key in ham:
    print(f"\nIdentity term ('{identity_key}'): {ham[identity_key]:.6f}")

# Verify round-trip conversion is lossless
qubit_ham_roundtrip = csvqe_dict_to_qubit_op(ham)
diff = qubit_ham - qubit_ham_roundtrip
max_diff = max((abs(v) for v in diff.terms.values()), default=0.0)
print(f"\nRound-trip conversion max coefficient error: {max_diff:.2e}  (should be ~0)")

cs_vqe ham dict: 15 terms
Pauli string length (= n_qubits): 4

First 10 terms:
  'IIII': -0.09706627
  'ZIII': 0.17141283
  'IZII': 0.17141283
  'IIZI': -0.22343154
  'IIIZ': -0.22343154
  'ZZII': 0.16868898
  'YXXY': 0.04530262
  'YYXX': -0.04530262
  'XXYY': -0.04530262
  'XYYX': 0.04530262

Identity term ('IIII'): -0.097066

Round-trip conversion max coefficient error: 0.00e+00  (should be ~0)


---
## 4. Step 3 — Split into Noncontextual and Contextual Parts

### 4a. Contextuality check

A Hamiltonian is **contextual** if its Pauli terms cannot all be simultaneously assigned definite ±1 values consistent with their commutation/anticommutation relations. This is tested by `c.contextualQ_ham(ham)`.  

Most molecular Hamiltonians beyond minimal H₂ are contextual.

### 4b. Finding the noncontextual sub-Hamiltonian

`c.greedy_dfs(ham, seconds, criterion)` performs a depth-first search to find the largest noncontextual subset of terms.  
- `criterion='weight'` maximises the sum of |coefficients| of included terms (recommended: captures most of the energy)
- `criterion='size'` maximises the number of included terms

**Output:** A list of noncontextual Pauli string subsets. The last element is the best found.

### 4c. Deriving the contextual part

`ham_con` = terms in `ham` that are **not** in `ham_noncon`.  
This is the part that requires quantum treatment.

In [5]:
# ─── 4a. Contextuality Check ──────────────────────────────────────────────────
is_contextual = c.contextualQ_ham(ham)
print(f"Is the full Hamiltonian contextual? {is_contextual}")

if not is_contextual:
    print("\nThe Hamiltonian is already noncontextual — no splitting needed.")
    print("All terms belong to the noncontextual part.")
    ham_noncon = dict(ham)
    ham_con = {}
else:
    print("Contextual Hamiltonian detected. Proceeding with CS-VQE decomposition.")

Is the full Hamiltonian contextual? False

The Hamiltonian is already noncontextual — no splitting needed.
All terms belong to the noncontextual part.


In [6]:
if is_contextual:
    # ─── 4b. Greedy DFS to find noncontextual sub-Hamiltonian ─────────────────
    # Increase `search_seconds` for better results on larger Hamiltonians.
    search_seconds = 10

    print(f"Running greedy DFS (criterion='weight', up to {search_seconds}s)...")
    subsets = c.greedy_dfs(ham, search_seconds, criterion='weight')

    # The last subset in the list is the largest noncontextual subset found
    terms_noncon = subsets[-1]
    ham_noncon = {p: ham[p] for p in terms_noncon}

    print(f"\nNoncontextual subset found: {len(terms_noncon)} terms out of {len(ham)} total")
    print(f"Weight fraction captured: "
          f"{sum(abs(v) for v in ham_noncon.values()) / sum(abs(v) for v in ham.values()):.2%}")

    # Verify it's really noncontextual
    assert not c.contextualQ(terms_noncon), "BUG: noncontextual subset is still contextual!"
    assert not c.contextualQ_ham(ham_noncon), "BUG: noncontextual Hamiltonian is still contextual!"
    print("✓ Verified: noncontextual subset is truly noncontextual")

    # ─── 4c. Derive contextual part ───────────────────────────────────────────
    # ham_con = all terms NOT in the noncontextual subset
    terms_con = [p for p in ham if p not in ham_noncon]
    ham_con = {p: ham[p] for p in terms_con}

    print(f"\nContextual subset: {len(ham_con)} terms")
    print(f"  (= full Hamiltonian minus noncontextual part)")

In [7]:
# ─── Summary of the split ─────────────────────────────────────────────────────
total_weight = sum(abs(v) for v in ham.values())
noncon_weight = sum(abs(v) for v in ham_noncon.values())
con_weight = sum(abs(v) for v in ham_con.values())

print("=" * 55)
print("  Hamiltonian decomposition summary")
print("=" * 55)
print(f"  Full Hamiltonian     : {len(ham):>5} terms  (weight: {total_weight:.4f})")
print(f"  Noncontextual part   : {len(ham_noncon):>5} terms  (weight: {noncon_weight:.4f}, {noncon_weight/total_weight:.1%})")
print(f"  Contextual part      : {len(ham_con):>5} terms  (weight: {con_weight:.4f}, {con_weight/total_weight:.1%})")
print("=" * 55)

print("\nNoncontextual Pauli terms:")
for pauli_str, coeff in list(ham_noncon.items())[:10]:
    print(f"  '{pauli_str}': {coeff:.8f}")
if len(ham_noncon) > 10:
    print(f"  ... ({len(ham_noncon) - 10} more)")

print("\nContextual Pauli terms:")
for pauli_str, coeff in list(ham_con.items())[:10]:
    print(f"  '{pauli_str}': {coeff:.8f}")
if len(ham_con) > 10:
    print(f"  ... ({len(ham_con) - 10} more)")

  Hamiltonian decomposition summary
  Full Hamiltonian     :    15 terms  (weight: 1.9842)
  Noncontextual part   :    15 terms  (weight: 1.9842, 100.0%)
  Contextual part      :     0 terms  (weight: 0.0000, 0.0%)

Noncontextual Pauli terms:
  'IIII': -0.09706627
  'ZIII': 0.17141283
  'IZII': 0.17141283
  'IIZI': -0.22343154
  'IIIZ': -0.22343154
  'ZZII': 0.16868898
  'YXXY': 0.04530262
  'YYXX': -0.04530262
  'XXYY': -0.04530262
  'XYYX': 0.04530262
  ... (5 more)

Contextual Pauli terms:


---
## 5. Step 4 — Noncontextual Ground State (Classical Part)

The noncontextual ground state energy is estimated classically. This:
1. Builds a quasi-quantized model of `ham_noncon`
2. Optimizes over its classical parameters `(q⃗, r⃗)`

This step produces `ep_state` — the energy-minimizing parameter assignment — which defines the contextual correction subspace.

In [8]:
if is_contextual:
    # Build quasi-quantized noncontextual model
    model = c.quasi_model(ham_noncon)
    commuting_gens    = model[0]  # universally-commuting generators G
    anticommuting_gens = model[1] # anticommuting generator pairs {C_i1}
    term_reconstruction = model[2] # how each term is expressed in generators

    print(f"Noncontextual model generators:")
    print(f"  Commuting generators (G):      {commuting_gens}")
    print(f"  Anticommuting generators (Ci1): {anticommuting_gens}")

    # Build energy function form
    fn_form = c.energy_function_form(ham_noncon, model)
    n_q_params = fn_form[0]  # number of q_j parameters (=|G|, each ±1)
    n_r_params = fn_form[1]  # number of r_i parameters (=|Ci1|, unit vector)
    print(f"\nEnergy function parameters:")
    print(f"  q-params (commuting, ±1): {n_q_params}")
    print(f"  r-params (anticommuting, unit vector): {n_r_params}")

    # Minimize energy over noncontextual parameters (brute force + scipy)
    print("\nOptimizing noncontextual ground state energy...")
    gs_noncon_result = c.find_gs_noncon(ham_noncon)
    gs_noncon_energy = gs_noncon_result[0]
    ep_state = gs_noncon_result[1]  # optimal parameter assignment [q1,...,qN, r1,...,rM]

    print(f"Classical (noncontextual) ground state energy: {gs_noncon_energy:.6f} Ha")
    print(f"Optimal parameter state (ep_state): {ep_state}")
else:
    # If already noncontextual, build model anyway for completeness
    model = c.quasi_model(ham_noncon)
    fn_form = c.energy_function_form(ham_noncon, model)
    gs_noncon_result = c.find_gs_noncon(ham_noncon)
    gs_noncon_energy = gs_noncon_result[0]
    ep_state = gs_noncon_result[1]
    print(f"Noncontextual ground state energy: {gs_noncon_energy:.6f} Ha")

Noncontextual ground state energy: -1.137284 Ha


In [9]:
model

(['ZZII', 'IZZI', 'IIZZ'],
 ['YXXY', 'ZIII'],
 {'IIII': [[], [], 1],
  'ZIII': [[], ['ZIII'], 1],
  'IZII': [['ZZII'], ['ZIII'], 1],
  'IIZI': [['ZZII', 'IZZI'], ['ZIII'], 1],
  'IIIZ': [['ZZII', 'IZZI', 'IIZZ'], ['ZIII'], 1],
  'ZZII': [['ZZII'], [], 1],
  'YXXY': [[], ['YXXY'], 1],
  'YYXX': [['IZZI', 'IIZZ'], ['YXXY'], (1+0j)],
  'XXYY': [['ZZII', 'IZZI'], ['YXXY'], (1+0j)],
  'XYYX': [['ZZII', 'IIZZ'], ['YXXY'], (1+0j)],
  'ZIZI': [['ZZII', 'IZZI'], [], 1],
  'ZIIZ': [['ZZII', 'IZZI', 'IIZZ'], [], 1],
  'IZZI': [['IZZI'], [], 1],
  'IZIZ': [['IZZI', 'IIZZ'], [], 1],
  'IIZZ': [['IIZZ'], [], 1]})

---
## 6. Step 4 (continued) — Get CS-VQE Reduced Hamiltonians

`get_reduced_hamiltonians` produces a list of **reduced contextual-correction Hamiltonians**, one per number of qubits devoted to the quantum computer (0 to n_qubits).  

- Index 0: pure classical approximation (0 quantum qubits)
- Index k: uses k qubits on the quantum processor
- Index n_qubits: exact (full quantum treatment)

**The reduced Hamiltonians already include the noncontextual energy in their constant term**, so the ground state energy of each reduced Hamiltonian IS the CS-VQE approximation at that qubit count.

In [10]:
# Default qubit ordering: move qubits from index 0 upward into the quantum part
# For better results, use csvqe_approximations_heuristic to find the optimal order.
order = list(range(n_qubits))

print(f"Qubit order for contextual expansion: {order}")
print("Computing reduced Hamiltonians for k = 0, 1, ..., n_qubits...")

reduced_hamiltonians = c.get_reduced_hamiltonians(ham, model, fn_form, ep_state, order)

print(f"\nReduced Hamiltonians computed: {len(reduced_hamiltonians)}")
for k, rh in enumerate(reduced_hamiltonians):
    n_terms = len(rh)
    n_qubits_rh = len(next(iter(rh))) if rh else 0
    print(f"  k={k} qubits on QC: {n_terms:4d} terms on {n_qubits_rh} qubits")

Qubit order for contextual expansion: [0, 1, 2, 3]
Computing reduced Hamiltonians for k = 0, 1, ..., n_qubits...

Reduced Hamiltonians computed: 5
  k=0 qubits on QC:    1 terms on 0 qubits
  k=1 qubits on QC:    2 terms on 1 qubits
  k=2 qubits on QC:    4 terms on 2 qubits
  k=3 qubits on QC:    8 terms on 3 qubits
  k=4 qubits on QC:   19 terms on 4 qubits


---
## 7. Step 5 — Extract the Contextual Reduced Hamiltonian

The **contextual reduced Hamiltonian** is the one you will use for downstream VQE.

Choose `k_qubits` — the number of qubits you want to allocate to the quantum computer:
- Larger `k_qubits` → better accuracy, more quantum resources needed
- Smaller `k_qubits` → less accurate, but classically tractable

The reduced Hamiltonian at index `k` operates on exactly `k` qubits (the others have been classically projected out).

In [11]:
# ─── Choose how many qubits to allocate to the quantum computer ───────────────
# Set k_qubits to the number of quantum qubits you want for the contextual correction.
# Must be in range [1, n_qubits]. Using n_qubits gives the exact result.
k_qubits = min(2, n_qubits)  # Example: 2 quantum qubits (adjust as needed)
# ─────────────────────────────────────────────────────────────────────────────

reduced_ham_dict = reduced_hamiltonians[k_qubits]

print(f"Selected reduced Hamiltonian: k={k_qubits} qubits on quantum processor")
print(f"  Number of terms: {len(reduced_ham_dict)}")
if reduced_ham_dict:
    k_actual = len(next(iter(reduced_ham_dict)))
    print(f"  Pauli string length (qubits in reduced space): {k_actual}")
print("\nReduced Hamiltonian terms:")
for pauli_str, coeff in reduced_ham_dict.items():
    print(f"  '{pauli_str}': {coeff:.8f}")

Selected reduced Hamiltonian: k=2 qubits on quantum processor
  Number of terms: 4
  Pauli string length (qubits in reduced space): 2

Reduced Hamiltonian terms:
  'II': -0.09985623
  'ZI': 0.00851385
  'ZZ': 0.73222839
  'IZ': -0.29668537


---
## 8. Step 6 — Convert Reduced Hamiltonian Back to OpenFermion Formats

We now convert `reduced_ham_dict` back to:
- **`con_qubit_ham`**: a `QubitOperator` — the reduced qubit Hamiltonian for use in VQE
- **`con_fermion_ham`**: a `FermionOperator` via inverse Jordan-Wigner — useful for active-space analysis

**Important note on `con_fermion_ham`:**  
The inverse Jordan-Wigner transform (`reverse_jordan_wigner`) only reconstructs a fermionic operator if the qubit operator's structure is compatible with JW encoding. The reduced Hamiltonian has been rotated by cs_vqe into a new basis, so the resulting `FermionOperator` should be treated as a **formal representation** for downstream software that requires fermionic input — not as a physically meaningful second-quantized operator in the original molecular orbital basis.

In [12]:
# ─── Convert reduced dict → QubitOperator ────────────────────────────────────
con_qubit_ham = csvqe_dict_to_qubit_op(reduced_ham_dict)
con_qubit_ham.compress()  # remove near-zero terms

n_qubits_reduced = count_qubits(con_qubit_ham)
print(f"Reduced QubitOperator:")
print(f"  {len(con_qubit_ham.terms)} terms on {n_qubits_reduced} qubits")
print(f"  {con_qubit_ham}"[:500])
print()

Reduced QubitOperator:
  4 terms on 2 qubits
  -0.09985622582524084 [] +
0.008513852076613393 [Z0] +
0.7322283910457872 [Z0 Z1] +
-0.29668536554086045 [Z1]



In [13]:
# ─── Convert reduced QubitOperator → FermionOperator ─────────────────────────
# Note: this is a formal/algebraic inverse JW transform.
# The resulting FermionOperator lives in the ROTATED basis, not the original MO basis.

try:
    con_fermion_ham = reverse_jordan_wigner(con_qubit_ham)
    con_fermion_ham = of.normal_ordered(con_fermion_ham)  # put in normal order
    print(f"Reduced FermionOperator (normal ordered):")
    print(f"  {len(con_fermion_ham.terms)} terms")
    # Print a few terms
    for i, (term, coeff) in enumerate(list(con_fermion_ham.terms.items())[:8]):
        print(f"  {coeff:.6f} {term}")
    if len(con_fermion_ham.terms) > 8:
        print(f"  ... ({len(con_fermion_ham.terms) - 8} more terms)")
except Exception as e:
    print(f"Warning: reverse_jordan_wigner failed ({e}).")
    print("This can happen when the reduced Hamiltonian has non-JW-compatible structure.")
    print("Using con_qubit_ham (QubitOperator) as the primary output instead.")
    con_fermion_ham = None

Reduced FermionOperator (normal ordered):
  4 terms
  0.344201 ()
  -1.481484 ((0, 1), (0, 0))
  -0.871086 ((1, 1), (1, 0))
  -2.928914 ((1, 1), (0, 1), (1, 0), (0, 0))


---
## 9. Verification — Ground State Energies

Verify the pipeline is correct by computing exact ground state energies and comparing with CS-VQE approximations.

In [14]:
from scipy.sparse.linalg import eigsh

print("Computing exact ground state energies for verification...\n")

# Full Hamiltonian exact ground state
e_full_exact = compute_exact_ground_state_energy(qubit_ham)
print(f"Full Hamiltonian exact GS energy:            {e_full_exact:.6f} Ha")

# Noncontextual approximation
print(f"Noncontextual (classical) approximation:     {gs_noncon_energy:.6f} Ha")

# CS-VQE reduced Hamiltonian ground state (this IS the CS-VQE approximation)
if con_qubit_ham.terms:
    e_reduced = compute_exact_ground_state_energy(con_qubit_ham)
    print(f"CS-VQE reduced Ham GS energy (k={k_qubits}):        {e_reduced:.6f} Ha")
    print(f"  Error vs exact:                          {abs(e_reduced - e_full_exact)*1000:.3f} mHa")

# CS-VQE approximation series (classical simulation for all k)
print(f"\nCS-VQE approximation accuracy vs qubit count (classical sim):")
print(f"  {'k':>4} | {'approx energy':>16} | {'error (mHa)':>12} | {'terms':>6}")
print(f"  {'─'*4}-+-{'─'*16}-+-{'─'*12}-+-{'─'*6}")

approx_series = c.contextual_subspace_approximations(ham, model, fn_form, ep_state, order)
for k, approx_e in enumerate(approx_series):
    error_mha = abs(approx_e - e_full_exact) * 1000
    n_terms_k = len(reduced_hamiltonians[k]) if k < len(reduced_hamiltonians) else '-'
    marker = ' ◀' if k == k_qubits else ''
    print(f"  {k:>4} | {approx_e:>16.6f} | {error_mha:>12.3f} | {n_terms_k:>6}{marker}")

print(f"\nChemical accuracy threshold: 1.6 mHa")

Computing exact ground state energies for verification...

Full Hamiltonian exact GS energy:            -1.137284 Ha
Noncontextual (classical) approximation:     -1.137284 Ha
CS-VQE reduced Ham GS energy (k=2):        -1.137284 Ha
  Error vs exact:                          0.000 mHa

CS-VQE approximation accuracy vs qubit count (classical sim):
     k |    approx energy |  error (mHa) |  terms
  ────-+-────────────────-+-────────────-+-──────
     0 |        -1.137284 |        0.000 |      1

Chemical accuracy threshold: 1.6 mHa


---
## 10. Final Output Summary

This cell collects all the key outputs for downstream use.

In [15]:
print("═" * 60)
print("  PIPELINE OUTPUT SUMMARY")
print("═" * 60)

print("\n── Step 1: Full Hamiltonian ──────────────────────────────")
print(f"  fermion_ham    : FermionOperator, {len(fermion_ham.terms)} terms")
print(f"  qubit_ham      : QubitOperator,   {len(qubit_ham.terms)} terms, {n_qubits} qubits")

print("\n── Step 2: Contextual Decomposition ─────────────────────")
print(f"  ham            : cs_vqe dict,     {len(ham)} terms (full)")
print(f"  ham_noncon     : cs_vqe dict,     {len(ham_noncon)} terms (noncontextual part)")
print(f"  ham_con        : cs_vqe dict,     {len(ham_con)} terms (contextual part)")

print("\n── Step 3: CS-VQE Reduced Hamiltonians ──────────────────")
print(f"  reduced_hamiltonians : list of {len(reduced_hamiltonians)} cs_vqe dicts")
print(f"  (index k = number of qubits allocated to QC)")

print(f"\n── Step 4: Reduced Output at k={k_qubits} (your downstream input) ──")
print(f"  reduced_ham_dict : cs_vqe dict,    {len(reduced_ham_dict)} terms")
print(f"  con_qubit_ham    : QubitOperator,  {len(con_qubit_ham.terms)} terms, {count_qubits(con_qubit_ham)} qubits")
if con_fermion_ham is not None:
    print(f"  con_fermion_ham  : FermionOperator, {len(con_fermion_ham.terms)} terms (rotated basis)")
else:
    print(f"  con_fermion_ham  : Not available (reverse JW failed — use con_qubit_ham)")

print("\n── Energy Summary ────────────────────────────────────────")
print(f"  Noncontextual classical approx : {gs_noncon_energy:.6f} Ha")
print(f"  CS-VQE (k={k_qubits}) GS energy      : {e_reduced:.6f} Ha")
print(f"  Exact GS energy (reference)    : {e_full_exact:.6f} Ha")
print(f"  CS-VQE error (k={k_qubits})          : {abs(e_reduced - e_full_exact)*1000:.3f} mHa")
print("═" * 60)

print("""
Next steps:
  • Use `con_qubit_ham` as input to your VQE ansatz
  • Use `con_fermion_ham` if your code expects a FermionOperator
  • Increase k_qubits (Section 7) to improve accuracy at the cost of more qubits
  • Use csvqe_approximations_heuristic() to find the optimal qubit ordering
""")

════════════════════════════════════════════════════════════
  PIPELINE OUTPUT SUMMARY
════════════════════════════════════════════════════════════

── Step 1: Full Hamiltonian ──────────────────────────────
  fermion_ham    : FermionOperator, 37 terms
  qubit_ham      : QubitOperator,   15 terms, 4 qubits

── Step 2: Contextual Decomposition ─────────────────────
  ham            : cs_vqe dict,     15 terms (full)
  ham_noncon     : cs_vqe dict,     15 terms (noncontextual part)
  ham_con        : cs_vqe dict,     0 terms (contextual part)

── Step 3: CS-VQE Reduced Hamiltonians ──────────────────
  reduced_hamiltonians : list of 5 cs_vqe dicts
  (index k = number of qubits allocated to QC)

── Step 4: Reduced Output at k=2 (your downstream input) ──
  reduced_ham_dict : cs_vqe dict,    4 terms
  con_qubit_ham    : QubitOperator,  4 terms, 2 qubits
  con_fermion_ham  : FermionOperator, 4 terms (rotated basis)

── Energy Summary ────────────────────────────────────────
  Noncontextual 

---
## 11. (Optional) Find Optimal Qubit Order via CS-VQE Heuristic

This cell uses the CS-VQE heuristic to find the optimal order to include qubits in the quantum part, minimising the CS-VQE error at each k.

⚠️ **Warning:** This takes **exponential time** in `n_qubits` (classical simulation). Safe for n_qubits ≤ 12.

In [16]:
# Only run if n_qubits is small enough
RUN_HEURISTIC = n_qubits <= 8  # set to True to force run on larger systems

if RUN_HEURISTIC:
    print(f"Running CS-VQE heuristic for optimal qubit ordering (n_qubits={n_qubits})...")
    csvqe_heuristic_result = c.csvqe_approximations_heuristic(
        ham, ham_noncon, n_qubits, e_full_exact
    )
    optimal_order = csvqe_heuristic_result[3]
    approx_series_opt = csvqe_heuristic_result[1]
    error_series_opt = csvqe_heuristic_result[2]

    print(f"\nOptimal qubit order: {optimal_order}")
    print(f"\n  {'k':>4} | {'approx energy':>16} | {'error (mHa)':>12}")
    print(f"  {'─'*4}-+-{'─'*16}-+-{'─'*12}")
    for k, (approx_e, err) in enumerate(zip(approx_series_opt, error_series_opt)):
        print(f"  {k:>4} | {approx_e:>16.6f} | {err*1000:>12.3f}")

    # Recompute reduced Hamiltonians with optimal order
    reduced_hamiltonians_opt = c.get_reduced_hamiltonians(
        ham, model, fn_form, ep_state, optimal_order
    )
    print(f"\nWith optimal order, reduced Hamiltonians recomputed.")
    print(f"To use them, access reduced_hamiltonians_opt[k] for k qubits on QC.")
else:
    print(f"Skipping heuristic (n_qubits={n_qubits} > 8).")
    print("Set RUN_HEURISTIC = True above to run anyway (may take a long time).")

Running CS-VQE heuristic for optimal qubit ordering (n_qubits=4)...

Optimal qubit order: [0, 1, 2, 3]

     k |    approx energy |  error (mHa)
  ────-+-────────────────-+-────────────
     0 |        -1.137284 |        0.000
     1 |        -1.137284 |        0.000
     2 |        -1.137284 |        0.000
     3 |        -1.137284 |        0.000

With optimal order, reduced Hamiltonians recomputed.
To use them, access reduced_hamiltonians_opt[k] for k qubits on QC.


---
## Appendix: Data Type Reference

### FermionOperator (OpenFermion)
```python
# Second-quantized fermionic operator
# Terms: {((mode_idx, action), ...): complex_coeff}
# action: 1 = creation (†), 0 = annihilation
FermionOperator('0^ 1', 0.5)   # 0.5 * a†_0 a_1
FermionOperator.terms  # {((0,1),(1,0)): 0.5}
```

### QubitOperator (OpenFermion)
```python
# Pauli operator on qubits
# Terms: {((qubit_idx, pauli_char), ...): complex_coeff}
QubitOperator('X0 Z2', 0.5)    # 0.5 * X_0 ⊗ I_1 ⊗ Z_2
QubitOperator.terms  # {((0,'X'),(2,'Z')): 0.5}
```

### cs_vqe Ham Dict
```python
# Pauli strings -> real coefficients
# String index = qubit index (left = qubit 0)
# Length must equal n_qubits for ALL entries
ham = {
    'IIII': -0.0989,    # identity
    'ZIII':  0.1714,    # Z on qubit 0
    'XXYY': -0.0453,    # X0 X1 Y2 Y3
}
```

### Conversion Summary
```python
# OpenFermion -> cs_vqe
ham = qubit_op_to_csvqe_dict(qubit_ham, n_qubits)

# cs_vqe -> OpenFermion QubitOperator
qubit_op = csvqe_dict_to_qubit_op(ham_dict)

# cs_vqe -> OpenFermion FermionOperator (via reverse JW)
fermion_op = reverse_jordan_wigner(csvqe_dict_to_qubit_op(ham_dict))
```